# Gold Layer — Aggregated Metrics
## SalesFlow Data Lakehouse | Phase 5: Analytical Layer

Creates pre-aggregated views on top of `fact_sales` and the dimension tables
for faster BI consumption. No new data is stored — views recompute on query.

| View | Grain | Description |
|---|---|---|
| `sales_by_month` | Year + Month | Total revenue, orders and quantity per month |
| `sales_by_category` | Product Category | Total

## 1. Sales by Month
Aggregates total revenue, number of distinct orders, and total quantity sold
per calendar year and month.

In [0]:
%sql
CREATE OR REPLACE VIEW salesflow_dev.gold.sales_by_month AS
SELECT
    d.year,
    d.month,
    d.month_name,
    COUNT(DISTINCT f.order_id)      AS total_orders,
    SUM(f.quantity)                 AS total_quantity,
    ROUND(SUM(f.line_total), 2)     AS total_revenue,
    ROUND(AVG(f.line_total), 2)     AS avg_line_value
FROM salesflow_dev.gold.fact_sales f
JOIN salesflow_dev.gold.dim_date d
    ON f.date_key = d.date_key
GROUP BY
    d.year,
    d.month,
    d.month_name
ORDER BY
    d.year,
    d.month;

## 2. Sales by Category
Aggregates total revenue, number of distinct orders, and total quantity sold
per product category.

In [0]:
%sql
CREATE OR REPLACE VIEW salesflow_dev.gold.sales_by_category AS
SELECT
    p.category_name,
    COUNT(DISTINCT f.order_id)      AS total_orders,
    SUM(f.quantity)                 AS total_quantity,
    ROUND(SUM(f.line_total), 2)     AS total_revenue,
    ROUND(AVG(f.unit_price), 2)     AS avg_unit_price,
    ROUND(AVG(f.discount), 4)       AS avg_discount
FROM salesflow_dev.gold.fact_sales f
JOIN salesflow_dev.gold.dim_product p
    ON f.product_key = p.product_key
GROUP BY
    p.category_name
ORDER BY
    total_revenue DESC;

## 3. Sales by Country
Aggregates total revenue, number of distinct orders, and number of distinct
customers per customer country.

In [0]:
%sql
CREATE OR REPLACE VIEW salesflow_dev.gold.sales_by_country AS
SELECT
    COALESCE(c.country, 'Unknown')      AS country,
    COUNT(DISTINCT f.order_id)          AS total_orders,
    COUNT(DISTINCT f.customer_key)      AS total_customers,
    SUM(f.quantity)                     AS total_quantity,
    ROUND(SUM(f.line_total), 2)         AS total_revenue,
    ROUND(SUM(f.line_total) /
          COUNT(DISTINCT f.order_id), 2) AS avg_order_value
FROM salesflow_dev.gold.fact_sales f
LEFT JOIN salesflow_dev.gold.dim_customer c
    ON f.customer_key = c.customer_key
GROUP BY
    COALESCE(c.country, 'Unknown')
ORDER BY
    total_revenue DESC;

## 4. Validation
Preview each view to confirm they return data correctly.

In [0]:
%sql
-- Sales by month: expect one row per year/month combination
SELECT * FROM salesflow_dev.gold.sales_by_month
ORDER BY year, month;

In [0]:
%sql
-- Sales by category: expect one row per category
SELECT * FROM salesflow_dev.gold.sales_by_category;

In [0]:
%sql
-- Sales by country: expect one row per country
SELECT * FROM salesflow_dev.gold.sales_by_country;

## 5. Cross-View Sanity Check
Total revenue must be consistent across all three views —
any discrepancy indicates a join or filter issue.

In [0]:
%sql
-- Total revenue should match across all three views
SELECT 'sales_by_month'    AS source, ROUND(SUM(total_revenue), 2) AS total FROM salesflow_dev.gold.sales_by_month
UNION ALL
SELECT 'sales_by_category' AS source, ROUND(SUM(total_revenue), 2) AS total FROM salesflow_dev.gold.sales_by_category
UNION ALL
SELECT 'sales_by_country'  AS source, ROUND(SUM(total_revenue), 2) AS total FROM salesflow_dev.gold.sales_by_country;